## Setup

In [1]:
# notebooks/01_data_cleaning.ipynb
#
# Cleaning pipeline for the Steam 2024 — Top 1,500 Games by Revenue dataset.
#
# NOTE ON OUTLIERS: this dataset is a "top N by revenue" ranking, so it is
# heavily right-skewed by construction — a handful of hits (Black Myth:
# Wukong, HELLDIVERS 2, Palworld, ...) legitimately dwarf the rest. A
# standard IQR-based capping step would misidentify those exact rows as
# "outliers" and clip them all down to the same value, destroying the
# signal the dataset exists to show. So this version does NOT blanket-cap
# numeric columns. Instead it relies on the domain checks already enforced
# in src/data_loader.py's validate_schema() (no negative price/revenue,
# reviewScore in [0, 100], no duplicate steamId) and only logs anything
# that looks like a genuine data-entry problem, without altering values.

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from src.data_loader import DataLoader


## Load & Validate Raw Data

In [2]:
PROJECT_ROOT = Path.cwd().parent
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

# Load raw data
loader = DataLoader()
df = loader.load_raw_data()          # data/raw/Steam_2024_bestRevenue_1500.csv
loader.validate_schema()             # raises if anything is structurally wrong


INFO | Loaded Steam_2024_bestRevenue_1500.csv -> 1500 rows x 11 columns
INFO | Schema validation passed.


True

## Cleaning Pipeline

Defines `clean_data()`: fills missing values, drops duplicates, runs sanity checks, tidies dtypes, and adds a derived `revenue_per_copy` column.

In [3]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleaning pipeline for the Steam 2024 revenue dataset.
    Returns cleaned DataFrame; writes a log of every change made.
    """
    cleaned_df = df.copy()
    cleaning_log = []

    # 1. Handle Missing Values
    for col in cleaned_df.columns:
        missing = cleaned_df[col].isnull().sum()
        if missing == 0:
            continue
        if pd.api.types.is_numeric_dtype(cleaned_df[col]):
            median = cleaned_df[col].median()
            cleaned_df[col] = cleaned_df[col].fillna(median)
            cleaning_log.append(
                f"Filled {missing} missing values in '{col}' with median ({median})"
            )
        else:
            mode = cleaned_df[col].mode()
            if len(mode) > 0:
                cleaned_df[col] = cleaned_df[col].fillna(mode[0])
                cleaning_log.append(
                    f"Filled {missing} missing values in '{col}' with mode ('{mode[0]}')"
                )

    # 2. Remove Duplicates
    duplicates = cleaned_df.duplicated().sum()
    if duplicates > 0:
        cleaned_df = cleaned_df.drop_duplicates()
        cleaning_log.append(f"Removed {duplicates} duplicate rows")

    # 3. Sanity checks (log-only, no silent modification)
    # These mirror validate_schema()'s checks; since load already validated
    # the raw file, this section should normally find nothing — it exists
    # as a safety net in case clean_data() is ever called on other data.
    if (cleaned_df['price'] < 0).any():
        n = (cleaned_df['price'] < 0).sum()
        cleaning_log.append(f"WARNING: {n} row(s) have negative price (left unmodified)")

    if (cleaned_df['revenue'] < 0).any():
        n = (cleaned_df['revenue'] < 0).sum()
        cleaning_log.append(f"WARNING: {n} row(s) have negative revenue (left unmodified)")

    if not cleaned_df['reviewScore'].dropna().between(0, 100).all():
        cleaning_log.append("WARNING: some reviewScore values fall outside 0-100 (left unmodified)")

    # 4. Fix Data Types (light touch — only convert genuinely low-cardinality
    #    text columns to category; do NOT touch numeric columns' scale/values)
    low_cardinality_text_cols = [
        col for col in cleaned_df.select_dtypes(include=['object', 'string']).columns
        if cleaned_df[col].nunique() < 10
    ]
    for col in low_cardinality_text_cols:
        cleaned_df[col] = cleaned_df[col].astype('category')
        cleaning_log.append(f"Converted '{col}' to categorical")

    # 5. Derived columns (optional — comment out if you don't want these)
    if 'revenue' in cleaned_df.columns and 'copiesSold' in cleaned_df.columns:
        cleaned_df['revenue_per_copy'] = cleaned_df['revenue'] / cleaned_df['copiesSold']
        cleaning_log.append("Added 'revenue_per_copy' (revenue / copiesSold)")

    # NOTE: deliberately NOT capping/clipping copiesSold, revenue, price, or
    # avgPlaytime — see the notebook intro above for why blanket IQR capping
    # is the wrong tool for a "top N by revenue" dataset.

    # Log cleaning actions
    with open(reports_dir / 'cleaning_log.txt', 'w') as f:
        f.write("DATA CLEANING LOG\n")
        f.write("=" * 50 + "\n\n")
        for action in cleaning_log:
            f.write(f"- {action}\n")
        f.write(f"\nFinal shape: {cleaned_df.shape}\n")

    return cleaned_df


## Run Cleaning & Save Output

In [4]:
# Clean the data
df_clean = clean_data(df)

# Save cleaned data
loader.save_processed(df_clean, "cleaned_data.csv")

# Show summary of changes
print("\nData cleaning complete!")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print("Check reports/cleaning_log.txt for details")


INFO | Saved processed data -> /home/natetat/Projects/statistics-standards/data/processed/cleaned_data.csv



Data cleaning complete!
Original shape: (1500, 11)
Cleaned shape: (1500, 12)
Check reports/cleaning_log.txt for details
